# Reward Design Matters: RLHF Pipeline Demo

This notebook walks through the complete RLHF pipeline:
1. Baseline generation
2. Preference dataset creation
3. Reward model training (simplicity & balanced)
4. PPO training (simplicity & balanced)
5. Evaluation
6. Misalignment analysis
7. Trade-off plots

**Mode:** Set `RLHF_MODE` to `"quick"` for a fast demo or `"extended"` for longer training.

In [ ]:
import os
import sys

# Ensure the project root is on the Python path
PROJECT_ROOT = os.path.dirname(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Set mode ("quick" or "extended")
os.environ["RLHF_MODE"] = "quick"

from src.config import print_config
print_config()

## Stage 1 — Baseline Generation

In [ ]:
from src.generate_baseline import run_baseline

baseline_df, baseline_summary = run_baseline()
baseline_df.head()

In [ ]:
# Inspect a few baseline responses
for _, row in baseline_df.head(3).iterrows():
    print(f"PROMPT: {row['prompt']}")
    print(f"RESPONSE: {row['response'][:200]}")
    print(f"  readability={row['readability']:.1f}  completeness={row['completeness']:.2f}  length={row['response_length']}")
    print("-" * 60)

## Stage 2 — Preference Dataset

In [ ]:
from src.create_preference_data import run_create_preference_data

simp_df, bal_df = run_create_preference_data()
print(f"Simplicity pairs: {len(simp_df)}")
print(f"Balanced pairs:   {len(bal_df)}")
simp_df.head()

## Stage 3 — Train Reward Models

In [ ]:
from src.train_reward_model_simplicity import run as train_rm_simplicity
from src.train_reward_model_balanced import run as train_rm_balanced

print("=" * 50)
print("Training Simplicity Reward Model")
print("=" * 50)
rm_simp, hist_simp = train_rm_simplicity()

print("\n" + "=" * 50)
print("Training Balanced Reward Model")
print("=" * 50)
rm_bal, hist_bal = train_rm_balanced()

## Stage 4 — PPO Training

In [ ]:
from src.train_ppo_simplicity import run as train_ppo_simp

print("=" * 50)
print("PPO Training — Simplicity Reward")
print("=" * 50)
ppo_simp_log = train_ppo_simp()

In [ ]:
from src.train_ppo_balanced import run as train_ppo_bal

print("=" * 50)
print("PPO Training — Balanced Reward")
print("=" * 50)
ppo_bal_log = train_ppo_bal()

## Stage 5 — Evaluation

In [ ]:
from src.evaluate import run_evaluation

eval_full_df, eval_summary_df = run_evaluation()
eval_summary_df

In [ ]:
# Side-by-side comparison for one prompt
import pandas as pd
sbs = pd.read_csv(os.path.join(PROJECT_ROOT, "outputs", "eval_side_by_side.csv"))
print(sbs.iloc[0].to_string())

## Stage 6 — Misalignment Analysis

In [ ]:
from src.misalignment_analysis import run_misalignment_analysis

flags_df, misalignment_summary = run_misalignment_analysis()
print(f"\nTotal flagged instances: {len(flags_df)}")
if len(flags_df) > 0:
    print(flags_df["failure_type"].value_counts())

## Stage 7 — Trade-Off Plots

In [ ]:
from src.plotting import run_plotting

run_plotting()

In [ ]:
# Display the key trade-off plot inline
from IPython.display import Image, display

fig_dir = os.path.join(PROJECT_ROOT, "outputs", "figures")
for fname in ["readability_vs_completeness.png", "metric_bar_chart.png", "radar_chart.png"]:
    fpath = os.path.join(fig_dir, fname)
    if os.path.exists(fpath):
        print(f"\n--- {fname} ---")
        display(Image(filename=fpath, width=600))

## Summary

This notebook executed the full 7-stage RLHF pipeline. Key takeaways:

- **Simplicity-only alignment** improves readability but reduces completeness and can cause over-shortening.
- **Balanced alignment** produces a better trade-off between clarity and informational content.
- **Reward design matters** — different reward functions lead to qualitatively different model behaviours and failure modes.

See `reports/final_report.md` for the full analysis and `reports/misalignment_report.md` for failure-mode details.